In [3]:
"""
Gene dataset overlap analysis: Venn diagram + per-region Excel export.

Usage:
    Edit the CONFIG section below (input/output paths, column names) and run:
        python gene_venn.py
    or import and call main() from another script / notebook.

Requires: pandas, openpyxl, matplotlib, venn
    pip install pandas openpyxl matplotlib venn
"""

import itertools
import pandas as pd
import matplotlib.pyplot as plt
from venn import venn as draw_venn

# --------------------------------------------------------------------------
# CONFIG - edit these to match your files/columns
# --------------------------------------------------------------------------
INPUT_XLSX = "input.xlsx"          # path to your input spreadsheet
GENE_COL = "Gene"                  # name of the gene column
# Names of the 4 Yes/No dataset columns, in the order you want them plotted.
# Edit these to match your actual column headers.
DATASET_COLS = ["Potts", "Edwards", "Kulkarni", "Sowa"]

VENN_PNG_OUT = "venn_diagram.png"          # where to save the Venn diagram image
OVERLAP_XLSX_OUT = "venn_overlaps.xlsx"    # where to save the overlap tables
# --------------------------------------------------------------------------


def load_gene_sets(input_xlsx, gene_col, dataset_cols):
    """Load the spreadsheet and return {dataset_name: set(genes marked 'Yes')}."""
    df = pd.read_excel(input_xlsx)

    missing = [c for c in [gene_col] + dataset_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"Column(s) not found in {input_xlsx}: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    gene_sets = {}
    for col in dataset_cols:
        # Normalize case/whitespace so "Yes", "yes ", "YES" all match.
        is_yes = df[col].astype(str).str.strip().str.lower() == "yes"
        gene_sets[col] = set(df.loc[is_yes, gene_col].astype(str))

    return df, gene_sets


def plot_venn(gene_sets, png_out):
    """Draw and save a 4-set Venn diagram."""
    plt.figure(figsize=(8, 8))
    draw_venn(gene_sets)  # dict of {label: set} -> handles 2-6 sets
    plt.tight_layout()
    plt.savefig(png_out, dpi=200)
    plt.close()
    print(f"Venn diagram saved to {png_out}")


def build_overlap_tables(df, gene_col, gene_sets):
    """
    Compute the exact-overlap region for every non-empty combination of
    datasets (i.e. genes in exactly those datasets and none of the others).

    Returns a dict: {region_label: DataFrame of full rows for genes in that region}
    """
    dataset_names = list(gene_sets.keys())
    overlap_tables = {}

    for r in range(1, len(dataset_names) + 1):
        for combo in itertools.combinations(dataset_names, r):
            in_these = set.intersection(*(gene_sets[name] for name in combo))
            others = [name for name in dataset_names if name not in combo]
            in_others = set.union(*(gene_sets[name] for name in others)) if others else set()
            exact_region = in_these - in_others

            if not exact_region:
                continue  # skip empty regions, nothing to export

            label = " & ".join(combo)
            region_df = df[df[gene_col].astype(str).isin(exact_region)].copy()
            overlap_tables[label] = region_df

    return overlap_tables


def export_overlap_tables(overlap_tables, xlsx_out):
    """Write each overlap region to its own sheet in one Excel workbook."""
    with pd.ExcelWriter(xlsx_out, engine="openpyxl") as writer:
        # Summary sheet first: region -> gene count
        summary = pd.DataFrame(
            [(label, len(t)) for label, t in overlap_tables.items()],
            columns=["Region", "Gene Count"],
        ).sort_values("Gene Count", ascending=False)
        summary.to_excel(writer, sheet_name="Summary", index=False)

        for label, table in overlap_tables.items():
            # Excel sheet names: max 31 chars, no []:*?/\
            sheet_name = label.replace(" & ", "+")
            for bad_char in r'[]:*?/\\':
                sheet_name = sheet_name.replace(bad_char, "")
            sheet_name = sheet_name[:31]
            table.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"Overlap tables saved to {xlsx_out}")


def main():
    df, gene_sets = load_gene_sets(INPUT_XLSX, GENE_COL, DATASET_COLS)

    for name, genes in gene_sets.items():
        print(f"{name}: {len(genes)} genes marked Yes")

    plot_venn(gene_sets, VENN_PNG_OUT)

    overlap_tables = build_overlap_tables(df, GENE_COL, gene_sets)
    export_overlap_tables(overlap_tables, OVERLAP_XLSX_OUT)

    print("Done.")


if __name__ == "__main__":
    main()

Potts: 52 genes marked Yes
Edwards: 27 genes marked Yes
Kulkarni: 11 genes marked Yes
Sowa: 30 genes marked Yes
Venn diagram saved to venn_diagram.png
Overlap tables saved to venn_overlaps.xlsx
Done.


<Figure size 800x800 with 0 Axes>

In [1]:
"""
Gene dataset overlap analysis: Venn diagram + per-region Excel export.

Usage:
    Edit the CONFIG section below (input/output paths, column names) and run:
        python gene_venn.py
    or import and call main() from another script / notebook.

Requires: pandas, openpyxl, matplotlib, venn
    pip install pandas openpyxl matplotlib venn
"""

import itertools
import pandas as pd
import matplotlib.pyplot as plt
from venn import venn as draw_venn

# --------------------------------------------------------------------------
# CONFIG - edit these to match your files/columns
# --------------------------------------------------------------------------
INPUT_XLSX = "input.xlsx"          # path to your input spreadsheet
GENE_COL = "Gene"                  # name of the gene column
# Names of the 4 Yes/No dataset columns, in the order you want them plotted.
# Edit these to match your actual column headers.
DATASET_COLS = ["Potts", "Edwards", "Kulkarni", "Sowa"]

VENN_PNG_OUT = "venn_diagram.png"          # where to save the Venn diagram image
OVERLAP_XLSX_OUT = "venn_overlaps.xlsx"    # where to save the overlap tables

# Text sizes
REGION_LABEL_FONTSIZE = 20   # the count numbers inside each region of the diagram
LEGEND_FONTSIZE = 16         # the dataset name legend
# --------------------------------------------------------------------------


def load_gene_sets(input_xlsx, gene_col, dataset_cols):
    """Load the spreadsheet and return {dataset_name: set(genes marked 'Yes')}."""
    df = pd.read_excel(input_xlsx)

    missing = [c for c in [gene_col] + dataset_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"Column(s) not found in {input_xlsx}: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    gene_sets = {}
    for col in dataset_cols:
        # Normalize case/whitespace so "Yes", "yes ", "YES" all match.
        is_yes = df[col].astype(str).str.strip().str.lower() == "yes"
        gene_sets[col] = set(df.loc[is_yes, gene_col].astype(str))

    return df, gene_sets


def plot_venn(gene_sets, png_out):
    """Draw and save a 4-set Venn diagram."""
    plt.figure(figsize=(8, 8))
    ax = draw_venn(gene_sets, fontsize=REGION_LABEL_FONTSIZE)  # dict of {label: set} -> handles 2-6 sets

    # Bump up the legend (dataset name) text size too.
    legend = ax.get_legend()
    if legend is not None:
        for text in legend.get_texts():
            text.set_fontsize(LEGEND_FONTSIZE)

    plt.tight_layout()
    plt.savefig(png_out, dpi=200)
    plt.close()
    print(f"Venn diagram saved to {png_out}")


def build_overlap_tables(df, gene_col, gene_sets):
    """
    Compute the exact-overlap region for every non-empty combination of
    datasets (i.e. genes in exactly those datasets and none of the others).

    Returns a dict: {region_label: DataFrame of full rows for genes in that region}
    """
    dataset_names = list(gene_sets.keys())
    overlap_tables = {}

    for r in range(1, len(dataset_names) + 1):
        for combo in itertools.combinations(dataset_names, r):
            in_these = set.intersection(*(gene_sets[name] for name in combo))
            others = [name for name in dataset_names if name not in combo]
            in_others = set.union(*(gene_sets[name] for name in others)) if others else set()
            exact_region = in_these - in_others

            if not exact_region:
                continue  # skip empty regions, nothing to export

            label = " & ".join(combo)
            region_df = df[df[gene_col].astype(str).isin(exact_region)].copy()
            overlap_tables[label] = region_df

    return overlap_tables


def export_overlap_tables(overlap_tables, xlsx_out):
    """Write each overlap region to its own sheet in one Excel workbook."""
    with pd.ExcelWriter(xlsx_out, engine="openpyxl") as writer:
        # Summary sheet first: region -> gene count
        summary = pd.DataFrame(
            [(label, len(t)) for label, t in overlap_tables.items()],
            columns=["Region", "Gene Count"],
        ).sort_values("Gene Count", ascending=False)
        summary.to_excel(writer, sheet_name="Summary", index=False)

        for label, table in overlap_tables.items():
            # Excel sheet names: max 31 chars, no []:*?/\
            sheet_name = label.replace(" & ", "+")
            for bad_char in r'[]:*?/\\':
                sheet_name = sheet_name.replace(bad_char, "")
            sheet_name = sheet_name[:31]
            table.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"Overlap tables saved to {xlsx_out}")


def main():
    df, gene_sets = load_gene_sets(INPUT_XLSX, GENE_COL, DATASET_COLS)

    for name, genes in gene_sets.items():
        print(f"{name}: {len(genes)} genes marked Yes")

    plot_venn(gene_sets, VENN_PNG_OUT)

    overlap_tables = build_overlap_tables(df, GENE_COL, gene_sets)
    export_overlap_tables(overlap_tables, OVERLAP_XLSX_OUT)

    print("Done.")


if __name__ == "__main__":
    main()

Potts: 57 genes marked Yes
Edwards: 30 genes marked Yes
Kulkarni: 12 genes marked Yes
Sowa: 34 genes marked Yes
Venn diagram saved to venn_diagram.png
Overlap tables saved to venn_overlaps.xlsx
Done.


<Figure size 800x800 with 0 Axes>